# 08 · Hugging Face Transformers Basics

In plain English, **Hugging Face** is the "app store" for AI models, and the `transformers` library is the remote control that lets you download a model and use it in about three lines of Python. Almost every fine-tuning project you'll ever do — including LoRA, QLoRA, and the capstone at the end of this course — is built on the exact tools in this notebook. So instead of jumping straight into fine-tuning, we'll spend this lesson getting comfortable with the toolkit: how to grab a model, run it, feed it data, and even run a tiny training loop that you barely have to write yourself.

Everything here uses **small, free models** that run on a plain laptop CPU. No GPU required.

## What you'll learn

- The **Hugging Face ecosystem**: the `transformers`, `datasets`, and `accelerate` libraries, plus the **Hub** where free models live.
- `pipeline()` — the fastest way to run a model, with a **sentiment** example and a **text-generation** example.
- `AutoTokenizer` and `AutoModel...` — loading a model and tokenizer by name and running a **manual forward pass**, then turning raw `logits` into **probabilities** with `softmax` and a prediction with `argmax`.
- The `datasets` library — building a `Dataset` from a Python dict, tokenizing it with `.map()`, and splitting it with `.train_test_split()`.
- A minimal end-to-end **`Trainer`** example: the same training loop you wrote by hand in notebook 05, now automated.
- **Saving and loading** a model and tokenizer with `save_pretrained` / `from_pretrained`.

## Why this matters for fine-tuning

Fine-tuning means taking a model someone else already trained and nudging it to be good at *your* task. To do that you need to be able to:

1. **Load** a pretrained model and its tokenizer (`AutoModel...`, `AutoTokenizer`).
2. **Prepare** your data as a `Dataset` and **tokenize** it (`.map()`).
3. **Run a training loop** (`Trainer` + `TrainingArguments`).
4. **Save** the result so you can use it later (`save_pretrained`).

Those are the four pillars of *every* fine-tuning recipe in this course. The fancy techniques (LoRA, QLoRA) just swap in a cheaper training step in the middle — the loading, data-prep, and saving stay exactly the same as what you'll learn here. Get these basics solid and the advanced notebooks become easy.

## Setup

Run the cell below once. The `%pip install` line is **commented out** — uncomment it if you're on Google Colab or a fresh environment.

The first time you load each model, `transformers` **downloads it from the Hugging Face Hub** (the free model store) and caches it on disk. So the first run of a cell is slow; after that it's instant because the files are local.

In [ ]:
# Uncomment the next line on Colab or a fresh environment:
# %pip install transformers datasets torch

import torch                      # PyTorch: the math engine the models run on
from transformers import pipeline # the "easy button" for running models
import transformers, datasets     # so we can print their versions

print("transformers version:", transformers.__version__)
print("datasets version:    ", datasets.__version__)
print("torch version:       ", torch.__version__)
# The first time you import these it may take a few seconds. That's normal.

## 1. The Hugging Face ecosystem in one picture

You will hear three names constantly. Here's what each one does:

- **`transformers`** — gives you the **models** and **tokenizers**. This is the star of the show.
- **`datasets`** — loads and transforms your **data** efficiently (handles large files without filling up memory).
- **The Hub** (`huggingface.co`) — a free website hosting **hundreds of thousands of models and datasets**. When you write a name like `"distilgpt2"`, `transformers` quietly downloads it from the Hub.

A fourth library, **`accelerate`**, handles running on GPUs/multiple devices behind the scenes — you rarely call it directly, but `Trainer` uses it for you.

Think of it like cooking: `datasets` is your prepped ingredients, `transformers` is the recipe and the oven, the Hub is the grocery store, and `accelerate` is the sous-chef who knows which burner to use.

## 2. `pipeline()` — the fastest way to run a model

`pipeline()` bundles three steps into one object: **tokenize the input → run the model → format the output** into something human-readable. You pick a *task* (like `"sentiment-analysis"`) and optionally a *model name*, and you get back a function you can call on text.

Let's classify some sentences as positive or negative using a small, popular sentiment model.

In [ ]:
# Create a sentiment-analysis pipeline.
# We name the model explicitly so we always get the same small one.
sentiment = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
)

# Call it on a list of sentences. Output is a list of dicts.
results = sentiment([
    "I absolutely loved this movie!",
    "This was a complete waste of my time.",
])

for sentence, r in zip(["sentence 1", "sentence 2"], results):
    print(sentence, "->", r)

# Expected (numbers may vary slightly):
# sentence 1 -> {'label': 'POSITIVE', 'score': 0.9998...}
# sentence 2 -> {'label': 'NEGATIVE', 'score': 0.9997...}

**What this does:**

- `pipeline("sentiment-analysis", model=...)` downloads the model (first time only) and wraps it so it's ready to call.
- We pass a **list of strings**, so we get back a **list of dicts**, one per sentence.
- Each dict has a `label` (`"POSITIVE"` or `"NEGATIVE"`) and a `score` — the model's **confidence** between 0 and 1. A score near 1.0 means "very sure."

### ✏️ Exercise

Call `sentiment(...)` on three sentences of your own — try to write one that's clearly positive, one clearly negative, and one **mixed or sarcastic** (e.g. *"Oh great, another Monday."*). Print the results and see whether the mixed one fools the model.

In [ ]:
# Your turn:
my_sentences = [
    # "write one here",
    # "and another",
    # "and a tricky one",
]
# print(sentiment(my_sentences))

## 3. `pipeline()` for text generation

The same `pipeline()` helper can **generate** text. We'll use **`distilgpt2`** — a tiny, distilled version of GPT-2 that runs fine on a CPU. It's not smart (it's small!), but it's perfect for learning.

The key setting is **`max_new_tokens`**: how many *new* words/word-pieces to add after your prompt. Keep it small (like 20) so generation is fast and cheap.

In [ ]:
generator = pipeline("text-generation", model="distilgpt2")

# Generate a short continuation of a prompt.
out = generator(
    "In the future, artificial intelligence will",
    max_new_tokens=20,   # add at most 20 new tokens (keep small on CPU)
    num_return_sequences=1,  # just one continuation
)

print(out[0]["generated_text"])
# Expected: your prompt followed by ~20 tokens of (somewhat random) text.
# distilgpt2 is tiny, so the output is often silly. That's fine for learning.

**What this does:**

- `pipeline("text-generation", model="distilgpt2")` loads a small generative model.
- We give it a **prompt** string; it predicts likely next tokens one at a time.
- `max_new_tokens=20` caps how much new text it writes — the single most important knob for keeping CPU runs fast.
- The result is a list of dicts; `out[0]["generated_text"]` is the prompt **plus** the generated continuation.

> A **token** is a chunk of text — often a word or part of a word. "tokenization" is just splitting text into these chunks. We'll see the tokenizer up close next.

### ✏️ Exercise

Run the generator twice with the **same prompt** but `max_new_tokens=10` once and `max_new_tokens=40` once. Notice how the longer one takes more time. Then try a completely different prompt of your own.

In [ ]:
# Your turn:
# print(generator("My favorite recipe is", max_new_tokens=10)[0]["generated_text"])
# print(generator("My favorite recipe is", max_new_tokens=40)[0]["generated_text"])

## 4. Under the hood: `AutoTokenizer`

`pipeline()` is convenient, but to fine-tune you need to see the pieces it hides. The first piece is the **tokenizer**: it turns text into **numbers** the model can read, and turns numbers back into text.

`AutoTokenizer.from_pretrained("name")` automatically loads the *correct* tokenizer for a given model — you don't have to know which exact class it is.

In [ ]:
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(model_name)

text = "I love fine-tuning!"
encoded = tokenizer(text)          # turn text into model inputs
print("input_ids:     ", encoded["input_ids"])
print("attention_mask:", encoded["attention_mask"])

# Peek at how the text was split into pieces ("tokens"):
print("tokens:        ", tokenizer.tokenize(text))

# Turn the ids back into text to prove it's reversible:
print("decoded:       ", tokenizer.decode(encoded["input_ids"]))

**What this does:**

- `tokenizer(text)` returns a dict with two key arrays:
  - **`input_ids`** — each token replaced by its integer ID from the model's vocabulary.
  - **`attention_mask`** — a list of 1s (and 0s for padding) telling the model which positions are real text vs. filler. All 1s here because we have a single, unpadded sentence.
- `tokenizer.tokenize(text)` shows the actual pieces — notice things like `"fine"`, `"-"`, `"tuning"`, and special markers. Subword splitting is normal.
- `tokenizer.decode(...)` reverses ids back to text. You'll see extra `[CLS]` and `[SEP]` tokens the model adds automatically to mark the start and end.

### ✏️ Exercise

Tokenize a **long** sentence and a **single word**. Compare the lengths of their `input_ids`. Then call the tokenizer on **two sentences at once** with `tokenizer(["first sentence", "second longer sentence"], padding=True)` and look at how the shorter one gets padded so both rows are the same length.

In [ ]:
# Your turn:
# print(tokenizer("hello"))
# print(tokenizer(["hi", "a much longer sentence here"], padding=True))

## 5. Under the hood: `AutoModelForSequenceClassification` + a manual forward pass

Now the second piece: the **model**. There are many `Auto...` model classes; you pick one based on the task:

- `AutoModel` — gives raw hidden features (advanced; the "naked" model).
- `AutoModelForSequenceClassification` — adds a classification head (positive/negative).
- `AutoModelForCausalLM` — for text generation (GPT-style).

We'll load the sentiment model **directly** (no pipeline) and run a **forward pass** by hand. This is exactly what happens inside `pipeline()` and inside training.

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.eval()  # put the model in "evaluation" mode (no training behavior)

# 1) Tokenize, asking for PyTorch tensors with return_tensors="pt".
inputs = tokenizer("I love fine-tuning!", return_tensors="pt")

# 2) Run the model. torch.no_grad() saves memory since we're not training.
with torch.no_grad():
    outputs = model(**inputs)   # **inputs unpacks input_ids + attention_mask

# 3) The raw scores are called "logits" — one per class, not yet probabilities.
logits = outputs.logits
print("logits shape:", logits.shape)  # -> torch.Size([1, 2]): 1 sentence, 2 classes
print("logits:      ", logits)

**What this does:**

- `return_tensors="pt"` makes the tokenizer output **PyTorch tensors** (instead of plain lists), which is what the model expects.
- `model(**inputs)` is the **forward pass** — the actual computation. `**inputs` spreads the dict into keyword arguments `input_ids=...`, `attention_mask=...`.
- `outputs.logits` are **raw, unnormalized scores** — one number per class. Bigger = "more likely this class," but they aren't probabilities yet (they can be negative or above 1).

In [ ]:
# From logits -> probabilities -> a prediction:
# softmax over dim=-1 (the class dimension) turns logits into probabilities.
probs = torch.softmax(logits, dim=-1)
print("probabilities:", probs)              # e.g. [[0.0002, 0.9998]]

# argmax = index of the highest probability = predicted class id.
predicted_id = torch.argmax(probs, dim=-1).item()
print("predicted class id:", predicted_id)  # 0 or 1

# The model stores a map from id -> human label. Use it to name the prediction.
print("id2label map:", model.config.id2label)
print("PREDICTION  :", model.config.id2label[predicted_id])
# Expected: PREDICTION : POSITIVE

**What this does:**

- `torch.softmax(logits, dim=-1)` converts the two logits into two probabilities that sum to 1. `dim=-1` means "do it across the last dimension," which is the classes.
- `torch.argmax(..., dim=-1)` returns the **position** of the largest probability; `.item()` pulls the single number out of the tensor.
- `model.config.id2label` is a dict the model carries with it (e.g. `{0: "NEGATIVE", 1: "POSITIVE"}`) so we can print a word instead of a number.

This three-step dance — **logits → softmax → argmax** — is how *every* classification model gives you an answer, and it's what you'll evaluate after fine-tuning.

### ✏️ Exercise

Wrap the steps above into a small function `predict(text)` that returns the label string and its probability. Test it on a clearly negative sentence and confirm it returns `"NEGATIVE"`.

In [ ]:
# Your turn:
def predict(text):
    inputs = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)
    pred_id = torch.argmax(probs, dim=-1).item()
    # return label and confidence:
    return model.config.id2label[pred_id], probs[0, pred_id].item()

# print(predict("This is the worst thing I have ever seen."))

## 6. The `datasets` library: building and transforming data

For fine-tuning you need your examples in a `Dataset` object. You *can* download ready-made ones from the Hub like this:

```python
from datasets import load_dataset
imdb = load_dataset("imdb", split="train[:1%]")  # just 1% of the data
```

But that needs an internet download. To stay **fully offline and fast**, we'll instead build a `Dataset` from a plain **Python dictionary** — which is exactly how you'll load your own custom data in real projects.

In [ ]:
from datasets import Dataset

# A Dataset from a dict: keys become COLUMNS, lists become the rows.
data = {
    "text": [
        "I loved this film, it was wonderful.",
        "Absolutely terrible, I want my money back.",
        "Best purchase of the year!",
        "Boring and far too long.",
        "A delightful and charming little story.",
        "Complete garbage, do not buy.",
    ],
    "label": [1, 0, 1, 0, 1, 0],  # 1 = positive, 0 = negative
}

ds = Dataset.from_dict(data)
print(ds)                  # shows columns and number of rows
print("one example:", ds[0])   # -> {'text': '...', 'label': 1}
print("number of rows:", len(ds))

**What this does:**

- `Dataset.from_dict(data)` builds a `Dataset` where each key (`"text"`, `"label"`) is a **column** and the lists supply the **rows**. Here we have 6 labeled examples.
- Printing `ds` shows its schema (`features`) and row count. `ds[0]` grabs the first row as a dict.
- This is the same shape `load_dataset(...)` returns, so anything you learn here works on Hub datasets too.

### Tokenizing with `.map()`

`.map()` runs a function over **every row** of the dataset and adds the results as new columns. We use it to tokenize all the text at once — efficiently, and without writing a loop. After this we'll also **split** the data with `.train_test_split()`.

In [ ]:
def tokenize_batch(batch):
    # padding/truncation keep every example the same length (max 32 tokens).
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=32,
    )

# batched=True hands many rows at once -> much faster.
tokenized = ds.map(tokenize_batch, batched=True)

print("columns now:", tokenized.column_names)
# -> ['text', 'label', 'input_ids', 'attention_mask']
print("first input_ids length:", len(tokenized[0]["input_ids"]))  # -> 32

**What this does:**

- `tokenize_batch` calls the tokenizer on a *batch* of texts. `padding="max_length"` + `max_length=32` make every row exactly 32 tokens; `truncation=True` cuts off anything longer.
- `ds.map(..., batched=True)` applies it to the whole dataset and **adds** `input_ids` and `attention_mask` as new columns, keeping the originals.
- Now the data has both the human-readable `text`/`label` and the model-ready numbers.

You always hold back some data to **test** on — otherwise you can't tell if the model actually learned or just memorized. `.train_test_split()` does this in one call.

In [ ]:
# Hold out a test set so we can measure real learning, not memorization:
split = tokenized.train_test_split(test_size=0.34, seed=42)
print(split)                       # a DatasetDict with 'train' and 'test'
print("train rows:", len(split["train"]))
print("test rows: ", len(split["test"]))

**What this does:**

- `test_size=0.34` puts ~1/3 of rows in the test set, the rest in train.
- `seed=42` makes the random split **reproducible** — you get the same split every run.
- The result is a `DatasetDict`: a dict-like object with `"train"` and `"test"` Datasets inside.

### ✏️ Exercise

Add **two more** examples (one positive, one negative) to the `data` dict, rebuild the `Dataset`, re-run `.map()`, and split again with `test_size=0.25`. Print the new train/test sizes.

In [ ]:
# Your turn: copy the data dict above, add two rows, and re-run the steps.

## 7. A minimal `Trainer`: the automated training loop

In notebook 05 you wrote a training loop by hand: forward pass → compute loss → backward pass → update weights, repeated for each batch and epoch. The `Trainer` class does **all of that for you**. You just hand it a model, the training arguments, and the data.

We'll do a *tiny* real fine-tune on our 6-example dataset with a small model, for **1 epoch**, just to prove the loop runs. This is the **same idea** you'll use for LoRA and the capstone — only the data and model get bigger.

> Heads up: this cell trains a real model on CPU. It may take **a minute or two**. That's expected.

In [ ]:
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

# Load a SMALL model fresh, set up for 2-class classification.
# "prajjwal1/bert-tiny" is a deliberately tiny BERT -> fast on CPU.
small_model = AutoModelForSequenceClassification.from_pretrained(
    "prajjwal1/bert-tiny",
    num_labels=2,
)
small_tok = AutoTokenizer.from_pretrained("prajjwal1/bert-tiny")

# Re-tokenize our data with THIS model's tokenizer.
def tok_fn(batch):
    return small_tok(batch["text"], padding="max_length",
                     truncation=True, max_length=32)

ds_tok = ds.map(tok_fn, batched=True)
ds_split = ds_tok.train_test_split(test_size=0.34, seed=42)
print("ready:", ds_split)

In [ ]:
# TrainingArguments = all the "knobs" for the run (where to save, how long, batch size).
args = TrainingArguments(
    output_dir="trainer_demo",     # folder for checkpoints/logs
    num_train_epochs=1,            # one pass over the data (keep tiny!)
    per_device_train_batch_size=2, # 2 examples at a time
    per_device_eval_batch_size=2,
    learning_rate=5e-4,
    logging_steps=1,
    report_to="none",              # don't try to log to external services
)

# Trainer ties model + args + data together and runs the loop for us.
trainer = Trainer(
    model=small_model,
    args=args,
    train_dataset=ds_split["train"],
    eval_dataset=ds_split["test"],
)

trainer.train()   # <-- the whole training loop, automated. Takes a minute or two.
print("Training finished!")

**What this does:**

- `TrainingArguments(...)` collects every setting for the run. The important ones for us: `num_train_epochs=1` (one quick pass) and the small batch sizes — both keep this CPU-friendly. `report_to="none"` just silences optional online logging.
- `Trainer(model=..., args=..., train_dataset=..., eval_dataset=...)` bundles everything.
- `trainer.train()` runs the **entire** forward/backward/update loop you wrote by hand in notebook 05 — for every batch, every epoch. You'll see a progress bar and a shrinking **loss**.

With only 6 examples and 1 epoch, the model won't actually get *good*. The point is simply: **the loop runs.** Scale up the data and epochs and this same code becomes real fine-tuning.

In [ ]:
# Evaluate on the held-out test set. Returns a dict with the loss.
metrics = trainer.evaluate()
print(metrics)   # e.g. {'eval_loss': 0.7..., 'eval_runtime': ...}

### ✏️ Exercise

Re-run the `TrainingArguments` and `Trainer` cells with `num_train_epochs=3` instead of 1. Watch the training **loss** in the logs — does it generally go down with more epochs? (On 6 examples it may be noisy; that's okay.)

## 8. Saving and loading your model

After fine-tuning you'll want to **keep** the result. Both models and tokenizers have a `save_pretrained(folder)` method that writes everything needed to disk, and a matching `from_pretrained(folder)` to load it back — the *same* method you used to load from the Hub, but pointed at a local folder.

In [ ]:
save_dir = "my_finetuned_model"

# Save BOTH the model and its tokenizer to the same folder.
small_model.save_pretrained(save_dir)
small_tok.save_pretrained(save_dir)

import os
print("saved files:", os.listdir(save_dir))
# You'll see things like config.json, model.safetensors, tokenizer files, etc.

**What this does:**

- `save_pretrained(save_dir)` writes the model's **config** (the architecture description) and **weights** (the learned numbers, usually a `model.safetensors` file).
- Saving the **tokenizer** to the same folder is essential — a model and its tokenizer must always travel together, or the input numbers won't mean anything to the model.

In [ ]:
# Load it back as if it were any other model on the Hub.
reloaded_model = AutoModelForSequenceClassification.from_pretrained(save_dir)
reloaded_tok = AutoTokenizer.from_pretrained(save_dir)

# Quick sanity check: run one prediction with the reloaded pair.
inputs = reloaded_tok("a wonderful experience", return_tensors="pt")
with torch.no_grad():
    logits = reloaded_model(**inputs).logits
print("reloaded prediction id:", torch.argmax(logits, dim=-1).item())
print("Reload successful — model and tokenizer restored from disk.")

**What this does:**

- `from_pretrained(save_dir)` reads the folder and rebuilds the exact model and tokenizer you saved.
- The sanity-check forward pass proves the reloaded model works just like the original. This is how you'll **share** or **deploy** a fine-tuned model later.

### ✏️ Exercise

Save your model to a **different** folder name (e.g. `"backup_model"`), then load it from there and run `predict`-style inference on one sentence. Confirm you get a class id back without errors.

## Common mistakes & how to debug them

- **"Why is the first run so slow?"** The model is **downloading from the Hub**. It's cached after that — the second run is fast. Not a bug.
- **Forgetting `return_tensors="pt"`.** If you feed the model plain Python lists instead of tensors, you'll get a type error. Always pass `return_tensors="pt"` when calling the model directly (the `pipeline` handles this for you).
- **Reading logits as probabilities.** Logits are raw scores and can be negative or large. Always apply `torch.softmax(logits, dim=-1)` before interpreting them as confidences.
- **Mismatched model and tokenizer.** Loading a tokenizer from one model and a model from another usually "runs" but gives nonsense. Keep each pair together; save them to the same folder.
- **Mixing up tasks.** `AutoModelForSequenceClassification` is for labels; `AutoModelForCausalLM` is for generation. Picking the wrong class triggers shape or head errors.
- **`Trainer` runs forever / kills your laptop.** You used too much data, too many epochs, or too big a model on CPU. Shrink everything: tiny model, few examples, `num_train_epochs=1`, small batch size.
- **`KeyError: 'labels'` in Trainer.** The `Trainer` expects the target column to be named **`label`** (or `labels`). Make sure your dataset has that column.

## Summary

- The Hugging Face stack is **`transformers`** (models + tokenizers), **`datasets`** (data), and the **Hub** (free downloads), with **`accelerate`** handling devices behind the scenes.
- **`pipeline()`** is the one-liner for instant inference — we ran **sentiment** and **text generation** with small models.
- Under the hood: **`AutoTokenizer`** turns text into `input_ids`, and an **`AutoModelFor...`** class runs a **forward pass** producing **logits**. Convert with **`softmax` → `argmax`** to get a prediction.
- **`datasets`**: build a `Dataset.from_dict(...)`, tokenize every row with **`.map()`**, and hold out a test set with **`.train_test_split()`**.
- **`Trainer` + `TrainingArguments`** automate the exact training loop from notebook 05 — we ran a tiny real fine-tune end-to-end on CPU.
- **`save_pretrained` / `from_pretrained`** persist and reload a model **and** its tokenizer (keep them together!).

These five skills — load, tokenize, forward pass, train, save — are the literal building blocks of every fine-tuning recipe ahead, including LoRA, QLoRA, and the capstone.

## What to learn next

Next up: **`09_dataset_preparation.ipynb`**. Now that you can *use* the tools, the next lesson goes deep on the part that makes or breaks fine-tuning — **preparing high-quality data**: cleaning text, formatting examples (prompt/response, JSONL), building train/validation splits, and getting everything into a `Dataset` that's ready to train on.